# Huấn luyện Mô hình Dự đoán Churn
Notebook này dùng để xây dựng và đánh giá mô hình học máy dự đoán khả năng rời mạng của khách hàng dựa trên dữ liệu có sẵn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

## 1. Chuẩn bị dữ liệu

In [ ]:
df = pd.read_csv('data/real_phase1_output.csv')

# Giả định churn = 1 nếu churn_risk_score > 0.5
df['is_churn'] = (df['churn_risk_score'] > 0.5).astype(int)

print(f"Số lượng khách hàng rời mạng / ở lại:")
print(df['is_churn'].value_counts())

# Bỏ các cột không dùng làm features và cả ltv/churn_risk_score vì chúng có thể là mục tiêu hoặc thông tin rò rỉ
features = df.drop(columns=['customer_id', 'name', 'churn_risk_score', 'ltv', 'is_churn'])
target = df['is_churn']

features.head()

## 2. Tiền xử lý dữ liệu (Pre-processing)

In [ ]:
# Phân loại cột thành số và danh mục
numeric_features = ['tenure_months', 'monthly_fee', 'total_charges']
categorical_features = ['contract_type', 'payment_method']

# Transformer cho các cột số
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Transformer cho các cột danh mục
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ghép lại thành một bộ tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 3. Xây dựng Pipeline & Huấn luyện mô hình Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)

# Khởi tạo mô hình Random Forest Classifier
rf_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

# Huấn luyện mô hình
rf_model.fit(X_train, y_train)
print("Hoàn tất huấn luyện mô hình!")

## 4. Đánh giá mô hình

In [ ]:
# Dự đoán trên tập test
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("Báo cáo phân loại (Classification Report):\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn (Confusion Matrix)')
plt.show()

In [ ]:
# ROC Curve và AUC Score
auc_score = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Tỷ lệ Dương tính giả (FPR)')
plt.ylabel('Tỷ lệ Dương tính thật (TPR)')
plt.title('Đường cong ROC')
plt.legend(loc='lower right')
plt.show()

## 5. Tầm quan trọng của các đặc trưng (Feature Importance)

In [ ]:
# Trích xuất feature importances từ mô hình
classifier = rf_model.named_steps['classifier']
importances = classifier.feature_importances_

# Trích xuất tên các features sau khi qua one-hot encoding
categorical_encoder = rf_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_feature_names = categorical_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

# Khởi tạo DataFrame lưu trữ
feat_imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Trực quan hóa
plt.figure(figsize=(10,6))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('Mức độ quan trọng của các đặc trưng (Feature Importance)')
plt.show()